# 7장. RAG 시스템 평가 — 06. Groundedness(근거성) 평가

## 학습 목표

- **Groundedness(근거성)**의 개념과 RAG 평가에서의 중요성 이해
- LLM 기반 근거성 평가자 구현
- 할루시네이션(Hallucination) 탐지 방법 학습

## 사전 지식

- 04번 노트북 커스텀 평가자 작성 경험
- Faithfulness 지표 이해 (02번 노트북)

## 평가 전략 의사결정 트리

어떤 평가 방식을 선택해야 할지 헷갈릴 때 참고하세요.

```mermaid
flowchart TD
    A[평가 시작] --> B{참조 답변<br/>있음?}
    B -->|있음| C{비용과<br/>정밀도 중 무엇이<br/>더 중요?}
    B -->|없음| D{빠른 스크리닝<br/>필요?}

    C -->|비용 절감| E[ROUGE / SemScore<br/>Heuristic 평가]
    C -->|정밀도 우선| F[LLM-as-Judge<br/>labeled_criteria]

    D -->|예| G[규칙 기반<br/>커스텀 평가자]
    D -->|아니오| H{컨텍스트가<br/>있는가?}

    H -->|있음| I[Groundedness 평가<br/>할루시네이션 탐지]
    H -->|없음| J[criteria<br/>참조 없는 LLM 평가]

    E --> K[LangSmith<br/>대시보드 분석]
    F --> K
    G --> K
    I --> K
    J --> K

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A input
    class B,C,D,H process
    class E,F,G,I,J process
    class K output
```

---

## 환경 설정

In [ ]:
# 필요한 패키지 설치
# !pip install -qU langsmith langchain langchain-openai


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(f"LANGCHAIN_API_KEY: {'✅ 설정됨' if os.getenv('LANGCHAIN_API_KEY') else '❌ 미설정'}")
print(f"LANGCHAIN_TRACING_V2: {os.getenv('LANGCHAIN_TRACING_V2', 'false')}")
print(f"LANGCHAIN_PROJECT: {os.getenv('LANGCHAIN_PROJECT', 'default')}")


---

## RAG 시스템 구축 (컨텍스트 포함 반환)

Groundedness 평가는 **답변**과 **검색된 컨텍스트**를 함께 비교해요. 따라서 평가용 함수가 컨텍스트도 함께 반환해야 합니다.

> **실무 팁**: 프롬프트에 "ONLY use the provided context"처럼 컨텍스트 기반 답변을 명시적으로 요구하면 Faithfulness 점수가 올라가요.

In [ ]:
# ---------------------------------------------------
# RAG 시스템 구축 (컨텍스트도 함께 반환하는 버전)
# ---------------------------------------------------
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1단계: 문서 로드 및 분할
loader = PyMuPDFLoader("data/attention_paper.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)

# 2단계: 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever()

# 3단계: 프롬프트 정의 — 컨텍스트 기반 답변 명시
prompt = PromptTemplate.from_template(
    """Answer the question based ONLY on the context provided. Do not add information not present in the context.

Context: {context}

Question: {question}

Answer:"""
)

# 4단계: LLM 및 체인 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Groundedness 평가를 위해 컨텍스트도 반환하는 함수
def rag_with_context(inputs: dict) -> dict:
    """RAG 답변과 함께 컨텍스트도 반환"""
    question = inputs["question"]
    
    # 컨텍스트 검색
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # 답변 생성
    answer = rag_chain.invoke(question)
    
    return {
        "question": question,
        "answer": answer,
        "context": context
    }

print("RAG 시스템 구축 완료")

---

## LLM 기반 Groundedness 평가자

LLM에게 "이 답변의 모든 내용이 제공된 컨텍스트에 근거하는가?"를 묻는 방식이에요.

- **grounded**: 답변의 모든 주장이 컨텍스트에서 확인 가능
- **not_grounded**: 컨텍스트에 없는 내용이 포함됨 (할루시네이션)

> **주의**: 평가자가 "grounded"를 포함하는 응답도 "not_grounded"와 혼동될 수 있어요. 예를 들어 "not_grounded"라는 응답에도 "grounded"가 포함되어 있으므로, `"not" not in result.lower()` 조건을 반드시 함께 확인합니다.

In [ ]:
# ---------------------------------------------------
# LLM 기반 Groundedness 평가자 구현
# ---------------------------------------------------
# ============================================================
# TODO: LLM에 "grounded" 또는 "not_grounded"를 판정하게 하는 평가자를 완성하세요
# 힌트: "grounded" in result.lower() and "not" not in result.lower() 로 이진 판정
# 예상 결과: {"key": "groundedness", "score": 0 또는 1}
# ============================================================

from langsmith.schemas import Run, Example
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import openai



---

## 데이터셋 준비 및 평가 실행

참조 답변 없이 질문만으로 평가해요. Groundedness는 `ground_truth`가 아닌 **검색된 컨텍스트**와 **생성된 답변**을 비교하므로 참조 답변이 없어도 됩니다.

In [ ]:
# ---------------------------------------------------
# LangSmith 데이터셋 준비 및 Groundedness 평가 실행
# ---------------------------------------------------
# ============================================================
# TODO: 질문만 있는 데이터셋을 만들고 llm_groundedness_evaluator로 evaluate()를 실행하세요
# 힌트: 참조 답변 없이 inputs만 있는 데이터셋도 가능 (Groundedness는 컨텍스트와 답변만 비교)
# 예상 결과: experiment_results.experiment_url 출력
# ============================================================

from langsmith import Client
from langsmith import utils as ls_utils
from langsmith.evaluation import evaluate

client = Client()
dataset_name = "transformer-qa-groundedness"

# 평가용 데이터 (참조 답변 없이 질문만)
qa_pairs = [
    {"question": "What is the Transformer architecture?"},
    {"question": "How does self-attention work?"},
    {"question": "What are the advantages of Transformers?"},
]



---

## 정리

### Groundedness가 RAG에서 중요한 이유

RAG 시스템에서 가장 위험한 실패는 **할루시네이션(Hallucination)**이에요. 검색된 문서에 없는 내용을 LLM이 생성하면 사용자에게 잘못된 정보를 전달하게 됩니다.

RAGAS의 **Faithfulness** 지표와 LangSmith의 **Groundedness** 평가자는 이를 서로 다른 방식으로 측정해요:

| 비교 항목 | RAGAS Faithfulness | LangSmith Groundedness |
|---------|-------------------|----------------------|
| 측정 방식 | 주장(claim) 단위 분해 후 검증 | LLM이 전체 답변 판단 |
| 출력 형식 | 0~1 연속 점수 | 0 또는 1 (이진 판단) |
| 비용 | LLM 호출 | LLM 호출 |
| 세밀도 | 높음 | 낮음 (빠른 스크리닝) |

### Groundedness 점수를 높이는 방법

1. **프롬프트 강화**: "제공된 컨텍스트에만 기반해서 답변하세요" 명시
2. **Temperature 낮추기**: 0으로 설정해 창의적 응답 줄이기
3. **검색 품질 개선**: 더 관련성 높은 문서를 찾아 LLM이 컨텍스트 밖으로 나가지 않도록
4. **더 강력한 LLM**: 지시를 더 잘 따르는 모델 사용

### 다음 노트북 예고

다음에는 **여러 모델을 동일한 데이터셋으로 비교**하는 방법을 배워요. GPT-4o-mini와 GPT-3.5-turbo를 같은 질문에 적용하고 LangSmith Compare 기능으로 나란히 비교해볼게요.